# BPR Matrix Factorization — 60/20/20 Experiment

## Overview

This notebook trains and evaluates a Bayesian Personalized Ranking (BPR) matrix factorization model using the new 60/20/20 chronological data split.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix
from implicit.bpr import BayesianPersonalizedRanking

# Project directory
project = Path(
    "/Users/ogechikaesedebe/Downloads/Activity-Recommendation-System"
)

# New dataset directory
split_dir = project / "data" / "splits_60_20_20"

# Check that the datasets exist
for filename in ["train.csv", "validation.csv", "test.csv"]:
    path = split_dir / filename

    print(
        f"{filename}:",
        "Found" if path.exists() else "Missing"
    )

print("\nLibraries imported successfully!")

train.csv: Found
validation.csv: Found
test.csv: Found

Libraries imported successfully!


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading the Training Dataset

In [2]:
# Load only the columns needed for BPR training
train_path = split_dir / "train.csv"

train_df = pd.read_csv(
    train_path,
    dtype={
        "user_id": "string",
        "gmap_id": "string"
    },
    usecols=["user_id", "gmap_id"]
)

print("TRAINING DATASET")

print("Total interactions:", len(train_df))
print("Unique users:", train_df["user_id"].nunique())
print("Unique activities:", train_df["gmap_id"].nunique())

print("\nFirst five interactions:")
display(train_df.head())

print("\nMissing values:")
print(train_df.isna().sum())

TRAINING DATASET
Total interactions: 6303944
Unique users: 627365
Unique activities: 69783

First five interactions:


,user_id,gmap_id
0,100000012795753291799,0x89c25d977025692d:0x378d24e13eb8f34d
1,100000012795753291799,0x89c258e28c304997:0xfcafe4e7ce35ee8c
2,100000012795753291799,0x89c2590f34e21a53:0x749079380908a830
3,100000012795753291799,0x89c25999b9c6eff9:0x4b98becb736928d8
4,100000012795753291799,0x89c25cfd883fc003:0xb4ba2be70609711b



Missing values:
user_id    0
gmap_id    0
dtype: int64


## Encoding User and Activity Identifiers

In [3]:
# Assign a numerical index to every unique user
user_indices, unique_users = pd.factorize(
    train_df["user_id"],
    sort=True
)

# Assign a numerical index to every unique activity
activity_indices, unique_activities = pd.factorize(
    train_df["gmap_id"],
    sort=True
)

# Create mappings from original IDs to numerical indices
user_to_index = {
    user_id: index
    for index, user_id in enumerate(unique_users)
}

activity_to_index = {
    activity_id: index
    for index, activity_id in enumerate(unique_activities)
}

# Create reverse mappings
index_to_user = {
    index: user_id
    for index, user_id in enumerate(unique_users)
}

index_to_activity = {
    index: activity_id
    for index, activity_id in enumerate(unique_activities)
}

# Record the number of users and activities
num_users = len(unique_users)
num_activities = len(unique_activities)

print("ENCODING RESULTS")
print("Number of users:", num_users)
print("Number of activities:", num_activities)

print("\nFirst five user indices:")
print(user_indices[:5])

print("\nFirst five activity indices:")
print(activity_indices[:5])

print("\nUser index range:")
print(user_indices.min(), "to", user_indices.max())

print("\nActivity index range:")
print(activity_indices.min(), "to", activity_indices.max())

ENCODING RESULTS
Number of users: 627365
Number of activities: 69783

First five user indices:
[0 0 0 0 0]

First five activity indices:
[22714  6900  8867 12868 22468]

User index range:
0 to 627364

Activity index range:
0 to 69782


## Constructing the User–Activity Interaction Matrix

In [4]:
# Construct the sparse user–activity interaction matrix
user_activity_matrix = csr_matrix(
    (
        np.ones(len(train_df), dtype=np.float32),
        (user_indices, activity_indices)
    ),
    shape=(num_users, num_activities),
    dtype=np.float32
)

# Calculate matrix sparsity
total_possible_interactions = num_users * num_activities

density = (
    user_activity_matrix.nnz
    / total_possible_interactions
) * 100

# Estimate memory usage
memory_mb = (
    user_activity_matrix.data.nbytes
    + user_activity_matrix.indices.nbytes
    + user_activity_matrix.indptr.nbytes
) / (1024 ** 2)

print("USER–ACTIVITY MATRIX")

print("Matrix shape:", user_activity_matrix.shape)
print("Observed interactions:", user_activity_matrix.nnz)

print(f"\nMatrix density: {density:.4f}%")
print(f"Matrix sparsity: {100 - density:.4f}%")

print(f"\nMemory usage: {memory_mb:.2f} MB")

print("\nMatrix format:", user_activity_matrix.getformat())

USER–ACTIVITY MATRIX
Matrix shape: (627365, 69783)
Observed interactions: 6303944

Matrix density: 0.0144%
Matrix sparsity: 99.9856%

Memory usage: 50.49 MB

Matrix format: csr


## Training 

In [5]:
import time

# Initialize the BPR model
bpr_model_60 = BayesianPersonalizedRanking(
    factors=64,
    learning_rate=0.01,
    regularization=0.01,
    iterations=50,
    random_state=42
)

print("Training BPR model on 60/20/20 dataset...")
print("Users:", num_users)
print("Activities:", num_activities)
print("Interactions:", user_activity_matrix.nnz)

# Measure training time
start_time = time.time()

# Train using the sparse interaction matrix
bpr_model_60.fit(
    user_activity_matrix,
    show_progress=True
)

training_time = time.time() - start_time

print("\nBPR training completed!")
print(f"Training time: {training_time:.2f} seconds")

print("\nLEARNED FACTOR SHAPES")
print("User factors:", bpr_model_60.user_factors.shape)
print("Activity factors:", bpr_model_60.item_factors.shape)

Training BPR model on 60/20/20 dataset...
Users: 627365
Activities: 69783
Interactions: 6303944


100%|█| 50/50 [03:27<00:00,  4.15s/it, train_auc=93.11%, skipped


BPR training completed!
Training time: 208.39 seconds

LEARNED FACTOR SHAPES
User factors: (627365, 65)
Activity factors: (69783, 65)


In [6]:
# Create the models directory if needed
models_dir = project / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Save the trained BPR model
model_path = models_dir / "bpr_60_20_20_50_iterations.npz"

bpr_model_60.save(str(model_path))

# Save the user and activity mappings
np.save(
    models_dir / "bpr_60_20_20_user_ids.npy",
    np.asarray(unique_users, dtype=str)
)

np.save(
    models_dir / "bpr_60_20_20_activity_ids.npy",
    np.asarray(unique_activities, dtype=str)
)

print("BPR model and mappings saved successfully!")

print("\nSaved files:")
for filename in [
    "bpr_60_20_20_50_iterations.npz",
    "bpr_60_20_20_user_ids.npy",
    "bpr_60_20_20_activity_ids.npy"
]:
    path = models_dir / filename
    print(f"{filename}: {'Found' if path.exists() else 'Missing'}")

BPR model and mappings saved successfully!

Saved files:
bpr_60_20_20_50_iterations.npz: Found
bpr_60_20_20_user_ids.npy: Found
bpr_60_20_20_activity_ids.npy: Found


## Preparing the Validation Dataset

In [7]:
# Load the 60/20/20 validation interactions
validation_path = split_dir / "validation.csv"

validation_df = pd.read_csv(
    validation_path,
    dtype={
        "user_id": "string",
        "gmap_id": "string"
    },
    usecols=["user_id", "gmap_id"]
)

# Check whether validation users and activities exist in training
unknown_users = ~validation_df["user_id"].isin(
    user_to_index
)

unknown_activities = ~validation_df["gmap_id"].isin(
    activity_to_index
)

print("VALIDATION DATASET")
print("Total interactions:", len(validation_df))
print("Unique users:", validation_df["user_id"].nunique())
print("Unique activities:", validation_df["gmap_id"].nunique())

print("\nCOLD-START CHECK")
print("Interactions with unknown users:", unknown_users.sum())
print("Interactions with unknown activities:", unknown_activities.sum())

print(
    "\nEvaluable interactions:",
    (~unknown_users & ~unknown_activities).sum()
)

print("\nFirst five validation interactions:")
display(validation_df.head())

VALIDATION DATASET
Total interactions: 1680489
Unique users: 553090
Unique activities: 67480

COLD-START CHECK
Interactions with unknown users: 0
Interactions with unknown activities: 0

Evaluable interactions: 1680489

First five validation interactions:


,user_id,gmap_id
0,100000012795753291799,0x89c245caca6f414d:0x9068bde30b6b8efe
1,100000036464530353686,0x89c283a3cd6cc823:0x73a889069a80a099
2,100000036464530353686,0x89c262aefaeb1751:0xf9be0dce89391eb7
3,100000036464530353686,0x89c27dfcb25a20af:0x5ded423833933c3f
4,100000036464530353686,0x89c261b248383c13:0xd06d170e1bb54a6f


## Evaluation Metrics

In [8]:
import math

def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    hits = len(set(recommended_k) & set(relevant))
    return hits / len(relevant)


def ndcg_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    dcg = 0
    for i, item in enumerate(recommended_k):
        if item in relevant:
            dcg += 1 / math.log2(i + 2)

    ideal_hits = min(len(relevant), k)

    idcg = sum(
        1 / math.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg


def hit_rate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    hits = set(recommended_k) & set(relevant)

    return 1 if len(hits) > 0 else 0


def calculate_metrics(recommended, relevant, k=10):
    return {
        f"Recall@{k}": recall_at_k(recommended, relevant, k),
        f"HitRate@{k}": hit_rate_at_k(recommended, relevant, k),
        f"NDCG@{k}": ndcg_at_k(recommended, relevant, k)
    }


print("Stacey's evaluation functions are ready!")

Stacey's evaluation functions are ready!


In [9]:
# Convert validation IDs into training-based numerical indices
validation_encoded = validation_df.copy()

validation_encoded["user_idx"] = (
    validation_encoded["user_id"].map(user_to_index)
)

validation_encoded["activity_idx"] = (
    validation_encoded["gmap_id"].map(activity_to_index)
)

# Confirm all IDs were mapped successfully
assert validation_encoded["user_idx"].notna().all()
assert validation_encoded["activity_idx"].notna().all()

# Group relevant validation activities by user
validation_ground_truth = (
    validation_encoded
    .groupby("user_idx")["activity_idx"]
    .apply(set)
    .to_dict()
)

# Calculate how many relevant activities each user has
relevant_counts = [
    len(activities)
    for activities in validation_ground_truth.values()
]

print("VALIDATION GROUND TRUTH")

print("Users with validation interactions:", len(validation_ground_truth))

print(
    "Average relevant activities per user:",
    round(np.mean(relevant_counts), 2)
)

print(
    "Median relevant activities per user:",
    np.median(relevant_counts)
)

print(
    "Maximum relevant activities for one user:",
    max(relevant_counts)
)

print("\nFirst three users and their relevant activity indices:")

for user_idx in list(validation_ground_truth.keys())[:3]:
    print(
        f"User {user_idx}:",
        sorted(validation_ground_truth[user_idx])[:10]
    )

VALIDATION GROUND TRUTH
Users with validation interactions: 553090
Average relevant activities per user: 3.04
Median relevant activities per user: 2.0
Maximum relevant activities for one user: 318

First three users and their relevant activity indices:
User 0: [3318]
User 1: [26010, 27517, 28020, 30826, 31935, 32003, 32499, 32520, 41563]
User 2: [12818, 13374]


In [10]:
# Get all users who have validation ground truth
eligible_user_indices = np.array(
    sorted(validation_ground_truth.keys()),
    dtype=np.int64
)

# Select 1,000 users reproducibly
rng = np.random.default_rng(42)

sample_size = min(1000, len(eligible_user_indices))

evaluation_user_indices = rng.choice(
    eligible_user_indices,
    size=sample_size,
    replace=False
)

# Convert numerical indices back to original user IDs
evaluation_user_ids = [
    index_to_user[int(user_idx)]
    for user_idx in evaluation_user_indices
]

# Save the shared evaluation sample
evaluation_path = (
    split_dir / "validation_sample_1000_users.csv"
)

pd.DataFrame({
    "user_id": evaluation_user_ids
}).to_csv(
    evaluation_path,
    index=False
)

print("EVALUATION USER SAMPLE")
print("Number of sampled users:", len(evaluation_user_ids))
print("Unique sampled users:", len(set(evaluation_user_ids)))

print("\nFirst five sampled user IDs:")
for user_id in evaluation_user_ids[:5]:
    print(user_id)

print("\nSaved evaluation sample to:")
print(evaluation_path)

EVALUATION USER SAMPLE
Number of sampled users: 1000
Unique sampled users: 1000

First five sampled user IDs:
113787016954182237620
114826022385978867624
108947677612734339932
112582663046175597692
106975861939900582533

Saved evaluation sample to:
/Users/ogechikaesedebe/Downloads/Activity-Recommendation-System/data/splits_60_20_20/validation_sample_1000_users.csv


In [11]:
# Store metric values for all evaluated users
metric_results = {
    "Recall@5": [],
    "Recall@10": [],
    "HitRate@5": [],
    "HitRate@10": [],
    "NDCG@5": [],
    "NDCG@10": []
}

for user_idx in evaluation_user_indices:
    user_idx = int(user_idx)

    # Generate top-10 BPR recommendations
    recommended_items, scores = bpr_model_60.recommend(
        userid=user_idx,
        user_items=user_activity_matrix[user_idx],
        N=10,
        filter_already_liked_items=True
    )

    # Relevant activities from validation
    relevant = validation_ground_truth[user_idx]

    # Calculate metrics at K=5 and K=10
    for k in [5, 10]:
        results = calculate_metrics(
            recommended=recommended_items,
            relevant=relevant,
            k=k
        )

        for metric_name, value in results.items():
            metric_results[metric_name].append(value)

# Average the metrics across users
average_metrics = {
    metric: np.mean(values)
    for metric, values in metric_results.items()
}

print("BPR VALIDATION RESULTS — 60/20/20")
print("Evaluated users:", len(evaluation_user_indices))

for metric, value in average_metrics.items():
    print(f"{metric}: {value:.6f}")

BPR VALIDATION RESULTS — 60/20/20
Evaluated users: 1000
Recall@5: 0.012833
Recall@10: 0.024212
HitRate@5: 0.036000
HitRate@10: 0.065000
NDCG@5: 0.011428
NDCG@10: 0.015270


In [12]:
# Prepare results for saving
results_df = pd.DataFrame([{
    "Model": "BPR",
    "Split": "60/20/20",
    "Evaluated Users": len(evaluation_user_indices),
    **average_metrics
}])

# Create results directory
results_dir = project / "results"
results_dir.mkdir(parents=True, exist_ok=True)

# Save validation results
results_path = results_dir / "bpr_validation_60_20_20.csv"
results_df.to_csv(results_path, index=False)

print("BPR validation results saved to:")
print(results_path)

display(results_df)

BPR validation results saved to:
/Users/ogechikaesedebe/Downloads/Activity-Recommendation-System/results/bpr_validation_60_20_20.csv


,Model,Split,Evaluated Users,Recall@5,Recall@10,HitRate@5,HitRate@10,NDCG@5,NDCG@10
0,BPR,60/20/20,1000,0.012833,0.024212,0.036,0.065,0.011428,0.01527
